In [36]:
pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.6/892.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 14.8 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9298 sha256=eecc06648a51a2df82ee90cf3eeaf7101675f6b097ab484f1d3e25dfd4d8f77a
  Stored in directory: /root/.cache/pip/wheels/bc/be/39/c37ad168eb2ff644c9685f52554440372129450f0b8ed203dd
Successfully built flashtext
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [49]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tf_keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv1D, Dropout, Dense, LSTM, MaxPooling1D, Bidirectional
from keras.optimizers import Adam
from sklearn.metrics import accuracy_score, classification_report
from keras.callbacks import EarlyStopping

 # load the dataset
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Train and test data preparation
texts_train = train_data["comment"].values
labels_train = train_data["label"].values

texts_test = test_data["comment"].values
labels_test = test_data["label"].values

# label Encoding
label_encoder = LabelEncoder()
labels_train = label_encoder.fit_transform(labels_train)
labels_test = label_encoder.transform(labels_test)

#Convert labels to one-hot
labels_train = to_categorical(labels_train, num_classes=2)
labels_test = to_categorical(labels_test, num_classes=2)

# Tokenizing and numerical sequences and padding  texts
max_no_tokens = 128
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts_train)

sequences_train = tokenizer.texts_to_sequences(texts_train)
sequences_test = tokenizer.texts_to_sequences(texts_test)

X_train = pad_sequences(sequences_train, maxlen=max_no_tokens)
X_test = pad_sequences(sequences_test, maxlen=max_no_tokens)

# Adding an extra dimension to Conv1D
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

#  LSTM model
model = Sequential()
model.add(Conv1D(32, kernel_size=3, activation='relu', padding='same', input_shape=(max_no_tokens, 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=['accuracy'])

# training model by Early Stopping
if os.path.exists("./lstm_trained_model_.h5"):
    print("مدل آموزش‌دیده بارگذاری شد.")
    model.load_weights("./lstm_trained_model_.h5")
else:
    print("مدل از ابتدا آموزش داده می‌شود.")
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model.fit(
        X_train, labels_train,
        epochs=10,
        batch_size=64,
        validation_split=0.2,
        verbose=1,
        callbacks=[early_stopping]
    )
    model.save("./lstm_trained_model_.h5")

# Model evaluation on test data
print("ارزیابی مدل روی داده‌های تست:")
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(labels_test, axis=1)

print("دقت مدل روی داده‌های تست:", accuracy_score(y_true_classes, y_pred_classes))
print("گزارش طبقه‌بندی:")
print(classification_report(y_true_classes, y_pred_classes, target_names=label_encoder.classes_))

test_loss, test_accuracy = model.evaluate(X_test, labels_test)
print(f"دقت روی داده تست: {test_accuracy:.2f}")


print(f"شکل X_test: {X_test.shape}")

# Prediction of new texts
new_texts = ["این محصول عالی بود!", "خیلی بد بود", "کیفیتش هم خیلی پایین‌تر از دفعات قبل", "قیمت خیلی مناسب است"]
new_sequences = tokenizer.texts_to_sequences(new_texts)

# Ensure that max_no_tokens is used
new_padded = pad_sequences(new_sequences, maxlen=max_no_tokens)
new_padded = np.expand_dims(new_padded, axis=-1)
print(f"شکل new_padded: {new_padded.shape}")

# predict
predictions = model.predict(new_padded)
predicted_classes = predictions.argmax(axis=1)

for text, label in zip(new_texts, predicted_classes):
    print(f"متن: {text} | احساس: {label}")


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


مدل آموزش‌دیده بارگذاری شد.
ارزیابی مدل روی داده‌های تست:
219/219 ━━━━━━━━━━━━━━━━━━━━ 14s 61ms/step
دقت مدل روی داده‌های تست: 0.626
گزارش طبقه‌بندی:
              precision    recall  f1-score   support

       HAPPY       0.67      0.51      0.58      3500
         SAD       0.60      0.75      0.67      3500

    accuracy                           0.63      7000
   macro avg       0.63      0.63      0.62      7000
weighted avg       0.63      0.63      0.62      7000

219/219 ━━━━━━━━━━━━━━━━━━━━ 14s 61ms/step - accuracy: 0.6221 - loss: 0.6385
دقت روی داده تست: 0.63
شکل X_test: (7000, 128, 1)
شکل new_padded: (4, 128, 1)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
متن: این محصول عالی بود! | احساس: 0
متن: خیلی بد بود | احساس: 1
متن: کیفیتش هم خیلی پایین‌تر از دفعات قبل | احساس: 1
متن: قیمت خیلی مناسب است | احساس: 0
